In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

In [ ]:
# Minkowski spacetime diagram visualiser

class Minkowski:

    def __init__(self, lightcone=False, hyperbolas=False, S0=True):
        """
        Initialise with lightcone and hyperbolas (to show path of Lorentz 
        boosts) as optional static plot elements.
        """

        # worldlines (or reference frames)
        self.worldlines = []

        # plot artists data
        self.axes = []
        self.gridlines = []
        self.labels = []
        
        # extent of plot units
        self.lim = 10

        # plot formatting
        plt.style.use("dark_background")
        self.fig, self.ax = plt.subplots(figsize=(6,6))
        plt.subplots_adjust(left=0.05, right=0.95, top=0.9, bottom=0.05)
        self.fig.suptitle(
            "Lorentz Boosts in Minkowski Spacetime", 
            fontsize=20
        )
        self.ax.set_xlim(-self.lim, self.lim)
        self.ax.set_ylim(-self.lim, self.lim)
        self.ax.get_xaxis().set_visible(False)
        self.ax.get_yaxis().set_visible(False)

        # speed relative to ground reference frame
        self.beta_text = self.ax.text(
            -self.lim, -self.lim-0.1,
            s=r"$\beta =  0.000$",
            fontsize=15,
            va="top"
        )

        # "present" dot to hide intersecting axes 
        self.ax.scatter(0, 0, s=50, color="grey", zorder=100)

        # format static plot elements
        if lightcone:
            # add 2 lightcones (45 degree lines)
            self.ax.plot(
                [-self.lim, self.lim], [-self.lim, self.lim], 
                linestyle="--", color="dimgrey", alpha=0.5
            )
            self.ax.plot(
                [-self.lim, self.lim], [self.lim, -self.lim],
                linestyle="--", color="dimgrey", alpha=0.5
            )
        
        if hyperbolas:
            # add 3 hyperbola branches per quarter
            x = np.linspace(-self.lim, self.lim, 200)
            for d in [3, 6, 9]:
                h = np.sqrt(d**2 + x**2)
                self.ax.plot(x,  h, linestyle="--", color="dimgrey", alpha=0.5)
                self.ax.plot(x, -h, linestyle="--", color="dimgrey", alpha=0.5)
                self.ax.plot(h,  x, linestyle="--", color="dimgrey", alpha=0.5)
                self.ax.plot(-h, x, linestyle="--", color="dimgrey", alpha=0.5)
                
        if S0:
            # add ground reference frame S0
            self.worldline("S0", 0, "grey", True, True)
        
    def worldline(
        self, name, beta, colour, grid, label, _built=False
    ):
        """
        Add a worldline (reference frame), specifying relative speed, beta, to 
        current ground frame as a fraction of c. _built ensure worldlines have 
        unique names or aren't defined multiple times.
        """
        
        if not _built:
            self.worldlines.append({
                "name":  name,   # identifier
                "beta":  beta,   # v / c
                "col" :  colour, # colour of axes/gridlines
                "grid":  grid,   # show gridlines
                "label": label   # show x and ct labels on axes
            })

        # offset factor to help plot axes/gridlines
        offset = self.lim * beta
        
        # plot ct and x axes as quivers
        self.axes.append(self.ax.quiver(
            -offset, -self.lim,
            2*offset, 2*self.lim,
            color=colour,
            angles="xy",
            scale_units="xy",
            scale=1,
            width=0.01,
            zorder=10
        ))
        
        self.axes.append(self.ax.quiver(
            -self.lim, -offset,
            2*self.lim, 2*offset,
            color=colour,
            angles="xy",
            scale_units="xy",
            scale=1,
            width=0.01,
            zorder=10
        ))

        if grid:
            # have gridlines evenly spaced
            for d0 in [-12, -9, -6, -3, 3, 6, 9, 12]:

                # contract d0 for correct gridlines
                d = d0 * np.sqrt(1 - beta**2)
                
                # constant position gridlines
                self.gridlines.append(self.ax.plot(
                    [d-offset, d+offset], [-self.lim, self.lim], 
                    color=colour, 
                    alpha=0.5
                )[0])
                
                # constant time gridlines
                self.gridlines.append(self.ax.plot(
                    [-self.lim, self.lim], [d-offset, d+offset], 
                    color=colour, 
                    alpha=0.5
                )[0])

        # x and ct labels on axes
        if label:

            # x label
            self.labels.append(self.ax.text(
                self.lim+0.1, offset, 
                s=r"$x$", 
                color=colour,
                fontsize=12,
                va="center"
            ))

            # ct label
            self.labels.append(self.ax.text(
                offset, self.lim+0.1,
                s=r"$ct$", 
                color=colour,
                fontsize=12,
                ha="center"
            ))

    def _redraw(self):
        """Clear and redraw diagram with new axes and gridlines."""

        # clear artists
        for artist in self.axes + self.gridlines + self.labels:
            artist.remove()
        self.axes.clear()
        self.gridlines.clear()
        self.labels.clear()

        # redraw artists
        for worldline in self.worldlines:
            self.worldline(*worldline.values(), _built=True)

    @staticmethod
    def _velocity_transform(beta1, beta2):
        """Perform velocity transformation from beta1 to beta2."""
        
        return (beta1 - beta2) / (1 - beta1 * beta2)

    @staticmethod
    def sign(beta):
        """Format sign of beta for plotting."""
        
        if round(beta, 3) >= 0:        
            return "+"    
        return "-"

    def boost(self, target, beta2=None):
        """
        Transform (boost) from current ground reference frame to target 
        reference frame.
        """

        # find speed of target reference frame
        if beta2 == None:
            for worldline in self.worldlines:
                if worldline["name"] == target:
                    beta2 = worldline["beta"]

        # update all worldline velocities
        for worldline in self.worldlines:
            worldline["beta"] = self._velocity_transform(
                worldline["beta"], beta2
            )

        # redraw diagram with boosted worldlines
        self._redraw()

        # update ground beta text
        self.beta_text.set_text(
            rf"$\beta = {self.sign(beta2)}{abs(beta2):.3f}$"
        )

    @staticmethod
    def _curve_out(vals):
        """
        Converts evenly-distributed values between 0 and 1 to be denser near the
        edges and sparser near the centre of the range.
        """
        return vals**2 / (2*vals**2 - 2*vals + 1)

    def _snapshot(self):
        """Store all current beta values."""
        return [worldline["beta"] for worldline in self.worldlines]

    def _restore(self, snapshot):
        """Restore all beta values to a given snapshot."""
        for worldline, beta in zip(self.worldlines, snapshot):
            worldline["beta"] = beta

    def animate(self, boosts, frames, save=False, title="Minkowski-diagram"):
        """Animate any sequence of Lorentz boosts."""

        # frames to allocate to each boost animation
        frames_per_boost = frames // len(boosts)

        # path to animate
        trajectory = []

        # current speeds relative to ground frame
        ground_betas = []

        # current speeds of all reference frames
        base_snapshot = self._snapshot()

        # base rapidity to kepp track of ground beta
        base_eta = 0

        # animate each boost sequence
        for target in boosts:

            # find boost target worldline index
            for i, worldline in enumerate(self.worldlines):
                if worldline["name"] == target:
                    target_index = i

            # find rapidity change
            target_beta = base_snapshot[target_index]
            eta2 = np.arctanh(target_beta)

            # interpolate rapidity
            for i in self._curve_out(np.linspace(0, 1, frames_per_boost)):

                eta_step = i * eta2
                beta_step = np.tanh(eta_step)

                # build all velocity snapshots for sequence
                state = [
                    self._velocity_transform(beta, beta_step)
                    for beta in base_snapshot
                ]
                
                trajectory.append(state)

                # keep track of ground beta values
                ground_beta = np.tanh(base_eta + eta_step)
                ground_betas.append(ground_beta)

            # animation starting point for next boost sequence
            base_snapshot = trajectory[-1].copy()

            base_eta += eta2

        # functions for FuncAnimation
        def init():
            self._restore(base_snapshot)
            self._redraw()
            return self.axes + self.gridlines
            
        def update(frame):
            self._restore(trajectory[frame])
            self._redraw()
            beta = ground_betas[frame]
            self.beta_text.set_text(
                rf"$\beta = {self.sign(beta)}{abs(beta):.3f}$"
            )

        # animate boosts sequence
        anim = FuncAnimation(
            self.fig,
            update,
            frames=frames,
            init_func=init,
            blit=False
        )

        if save:
            anim.save(f"{title}.gif", writer=PillowWriter(fps=frames/10))

        plt.close(self.fig)

        if not save:
            return HTML(anim.to_jshtml())